#Chapter 3 self attention



#Importing the tokenized stuff from tokenizer file

In [1]:
#OPENONG AND READING fILE
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()

    

In [2]:
# Now Byte pair encoding
#using tiktoken

import tiktoken
# version check for tiktoken
from importlib.metadata import version
print("tiktoken version:", version("tiktoken"))

# now actual byte pair encoding using tiktoken
TTokenizer=tiktoken.get_encoding("gpt2")

tiktoken version: 0.13.0


In [3]:

# Dataset and Dataloader-> Dataloader will return x and y in batches, so we can train the model on those batches

# using the pytorch dataset and dataloader

import torch
from torch.utils.data import Dataset, DataLoader


#making a dataset class

class LLMDatasetV1(Dataset):
#text is the whole text, tokenizer is the tiktoken tokenizer, max_length is the context size, stride is how many tokens to move forward for the next input
    def __init__(self,text,tokenizer,max_length,stride):
        self.input_ids=[]
        self.target_ids=[]

        token_ids=TTokenizer.encode(text)

        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk= token_ids[i:i+max_length]
            target_chunks=token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))  
            self.target_ids.append(torch.tensor(target_chunks))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
    



# Now the dataloader function 

def create_DataloaderV1(text, batch_size=4, max_length=256, stride=128,shuffle=True, drop_last=True, num_workers=0):
    
    dataset= LLMDatasetV1(text,TTokenizer,max_length,stride)
    dataLoader= DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,
                           drop_last=drop_last,num_workers=num_workers)
    return dataLoader




In [ ]:
# CALLING BOTH
max_length=4
dataloader= create_DataloaderV1(raw_text,batch_size=8,max_length=4,stride=4,shuffle=False)
data_iter=iter(dataloader)

inputs,targets = next(data_iter)
print("Inputs \n",inputs)
print("Targets \n",targets)

In [ ]:
# Conversion to embeddings -> we are using tiktoken'z gpt2 tokenizer and its vocab is 50257 words
Tvocab_size=50257
output_dim=256
token_embedding_layer=torch.nn.Embedding(Tvocab_size,output_dim)
# now we see the batch previously implemented
print("Inputs \n",inputs)
print("Inputs Shape \n",inputs.shape)

# Now the embeddings
token_embeddings=token_embedding_layer(inputs)
print(token_embeddings.shape)
print(token_embeddings)


In [ ]:
#positional embeddings
# now the positonal embeddings, we need another embedding layer
#max_length is 4
context_length= max_length
pos_embedding_layer=torch.nn.Embedding(context_length,output_dim)
pos_embeddings=pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

#add it to token embeddings 
input_embeddings= token_embeddings + pos_embeddings
print(input_embeddings.shape)


# NOW the Actual Chapter 3 --> Self attention

In [ ]:
#Self attention

# new text
stext= "Your Journey starts with one step"

# taking example tensor for this
import torch
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)

)

# we first need to get attention scores for each element with the query
#query is the input sequence/token we want to get the context vector for
#context vector is the enhanced embedded vector for a token, containing information with respect to other tokens

#taking token 2 as query--> journey

input_query=inputs[1]
input_1=inputs[0]

#now we need to take dot product
torch.dot(input_query,input_1)

#but we need to automate for getting this for all inputs relative to the input query

print(inputs.shape[0])
attention_score2=torch.empty(inputs.shape[0])

for i, xi in enumerate(inputs):
    attention_score2[i]=torch.dot(xi,input_query)   
print(attention_score2)


6
tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
